# Prédiction du prix médian des logements en Californie

## Contexte métier
L'objectif est de prédire `MedHouseVal`, la valeur médiane des logements d'un district
californien (en centaines de milliers de dollars), à partir de variables démographiques
et immobilières issues du recensement de 1990 (dataset `fetch_california_housing` de scikit-learn).

## Sommaire
1. [Chargement et aperçu des données](#1)
2. [Analyse exploratoire (EDA)](#2)
3. [Feature engineering](#3)
4. [Préparation train/test](#4)
5. [Modélisation : du baseline au modèle final](#5)
6. [Optimisation des hyperparamètres](#6)
7. [Interprétabilité (feature importance & SHAP)](#7)
8. [Analyse des erreurs](#8)
9. [Export pour déploiement](#9)
10. [Conclusion](#10)

## ⚠️ Limite connue du dataset
`MedHouseVal` est **plafonné à 5.00001** (soit 500 001 $) : toutes les valeurs réelles
au-delà de ce seuil ont été censurées lors de la collecte (recensement US de 1990).
Le modèle ne pourra donc jamais apprendre à prédire des valeurs plus élevées correctement,
et les erreurs sur les logements de luxe doivent être interprétées à la lumière de cet artefact,
pas comme une vraie faiblesse du modèle.

<a id='1'></a>
## 1. Chargement et aperçu des données

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_validate, RandomizedSearchCV
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import shap

RANDOM_STATE = 42  # graine fixée partout pour la reproductibilité des résultats
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)  # chemin relatif : fonctionne sur n'importe quelle machine

In [ ]:
housing = fetch_california_housing()

# Bunch -> DataFrame : plus pratique pour l'exploration et le feature engineering
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df["MedHouseVal"] = housing.target

df.head()

In [ ]:
print(f"Dimensions : {df.shape[0]} observations, {df.shape[1] - 1} variables explicatives")
df.info()

In [ ]:
# Vérification qualité : valeurs manquantes et doublons
print("Valeurs manquantes par colonne :")
print(df.isna().sum())
print(f"\nLignes dupliquées : {df.duplicated().sum()}")
# -> Aucune valeur manquante ni doublon : ce dataset est déjà nettoyé en amont par scikit-learn.

In [ ]:
df.describe()

<a id='2'></a>
## 2. Analyse exploratoire (EDA)

### 2.1 Distributions univariées

In [ ]:
df.hist(figsize=(12, 10), bins=30)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/histograms.png", dpi=150, bbox_inches="tight")
plt.show()

**Observations :**
- `MedInc` (revenu médian) a une distribution asymétrique à droite, typique des revenus.
- `AveRooms`, `AveBedrms`, `AveOccup` et `Population` présentent des valeurs extrêmes
  très éloignées du reste de la distribution (creusé en 2.2).
- `MedHouseVal` montre un pic net au plafond de 5.0 — c'est la censure mentionnée en introduction.

### 2.2 Détection des valeurs extrêmes (méthode IQR)

In [ ]:
features_to_check = ["MedInc", "HouseAge", "AveRooms", "AveBedrms",
                      "Population", "AveOccup"]

outlier_summary = []
for col in features_to_check:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_outliers = df[(df[col] < lower) | (df[col] > upper)].shape[0]
    outlier_summary.append({"feature": col, "lower_bound": lower,
                             "upper_bound": upper, "n_outliers": n_outliers,
                             "pct_outliers": round(100 * n_outliers / len(df), 2)})

pd.DataFrame(outlier_summary)

**Décision : ne pas supprimer ces outliers.**
Ils ne sont pas des erreurs de saisie mais des districts réellement atypiques (ex. logements
de vacances avec très peu d'occupants permanents). Les modèles à base d'arbres (Random Forest,
Gradient Boosting) sont naturellement robustes à ces valeurs extrêmes puisqu'ils partitionnent
l'espace par seuils plutôt que par distance — les supprimer ferait perdre de l'information
sans bénéfice clair pour ces familles de modèles.

### 2.3 Corrélations avec la variable cible

In [ ]:
correlation = df.corr(numeric_only=True)
correlation["MedHouseVal"].sort_values(ascending=False)

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matrice de corrélation")
plt.savefig(f"{OUTPUT_DIR}/correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

pairs = [("MedInc", "Median Income"), ("HouseAge", "House Age"),
         ("AveRooms", "Average Rooms"), ("AveOccup", "Average Occupancy")]

for ax, (col, label) in zip(axes.flatten(), pairs):
    ax.scatter(df[col], df["MedHouseVal"], alpha=0.2)
    ax.set_title(f"{label} vs Median House Value")
    ax.set_xlabel(label)
    ax.set_ylabel("Median House Value")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/scatter_relationships.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Vue géographique : les prix élevés se concentrent sur la côte (San Francisco, Los Angeles)
plt.figure(figsize=(10, 7))
scatter = plt.scatter(df["Longitude"], df["Latitude"], c=df["MedHouseVal"],
                       alpha=0.4, s=10, cmap="viridis")
plt.colorbar(scatter, label="Median House Value")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Répartition géographique de la valeur des logements")
plt.savefig(f"{OUTPUT_DIR}/geographic_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

<a id='3'></a>
## 3. Feature engineering

Les variables brutes (`AveRooms`, `AveBedrms`, `Population`, `AveOccup`) sont des agrégats
peu interprétables isolément. On construit des ratios plus parlants, une pratique standard
sur ce dataset et qui améliore généralement la performance des modèles linéaires :

- `rooms_per_household` : nombre moyen de pièces par logement
- `bedrooms_per_room` : proportion de chambres parmi les pièces (proxy de la taille du logement)
- `population_per_household` : densité d'occupation par logement (redondant avec `AveOccup`
  mais exprimé différemment, utile pour les modèles linéaires)

In [ ]:
df["rooms_per_household"] = df["AveRooms"] / df["AveOccup"]
df["bedrooms_per_room"] = df["AveBedrms"] / df["AveRooms"]
df["population_per_household"] = df["Population"] / df["AveOccup"]

# On vérifie l'absence de division par zéro / valeurs infinies introduites
new_features = ["rooms_per_household", "bedrooms_per_room", "population_per_household"]
print(df[new_features].isna().sum())
print(np.isinf(df[new_features]).sum())

<a id='4'></a>
## 4. Préparation train / test

In [ ]:
X = df.drop(columns="MedHouseVal")
y = df["MedHouseVal"]

# 80/20 : on conserve un jeu de test jamais vu pendant l'entraînement,
# seule façon fiable d'estimer la capacité de généralisation du modèle.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print(f"Train : {X_train.shape[0]} lignes | Test : {X_test.shape[0]} lignes")

<a id='5'></a>
## 5. Modélisation : du baseline au modèle final

Pour éviter de répéter le calcul des métriques à chaque modèle, on factorise
l'évaluation dans une fonction réutilisable.

In [ ]:
def evaluate(name, y_true, y_pred, results_list):
    """Calcule MAE, RMSE, R² et les ajoute à la liste de résultats pour comparaison finale."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)

    print(f"{name:<20} MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}")
    results_list.append({"Model": name, "MAE": mae, "RMSE": rmse, "R2": r2})
    return mae, rmse, r2


results = []

### 5.1 Baseline (référence minimale)

In [ ]:
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
y_pred_baseline = baseline.predict(X_test)

evaluate("Baseline", y_test, y_pred_baseline, results)
# Un modèle utile doit impérativement faire mieux que cette référence triviale.

### 5.2 Régression linéaire

In [ ]:
model_lr = LinearRegression()
model_lr.fit(X_train, y_train)
y_pred_lr = model_lr.predict(X_test)

evaluate("Linear Regression", y_test, y_pred_lr, results)

### 5.3 Ridge Regression

La régularisation L2 de Ridge pénalise les coefficients trop élevés, ce qui limite le
surapprentissage en présence de variables corrélées (`AveRooms`/`rooms_per_household` par
exemple). **Le `StandardScaler` est indispensable ici** : Ridge pénalise les coefficients
selon leur magnitude, donc des variables à des échelles très différentes (`Population` en
milliers vs `MedInc` en dizaines) fausseraient la régularisation sans normalisation préalable.
Les modèles à base d'arbres (section suivante) n'ont pas besoin de ce scaling, car ils ne
comparent jamais les variables entre elles — ils ne font que des seuils sur une seule
variable à la fois.

In [ ]:
ridge_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0)),
])

ridge_pipeline.fit(X_train, y_train)
y_pred_ridge = ridge_pipeline.predict(X_test)

evaluate("Ridge", y_test, y_pred_ridge, results)

### 5.4 Random Forest

In [ ]:
rf_model = RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

evaluate("Random Forest", y_test, y_pred_rf, results)

### 5.5 Gradient Boosting

À titre de comparaison avec un autre ensemble d'arbres, utilisant cette fois du boosting
séquentiel plutôt que du bagging parallèle.

In [ ]:
gb_model = GradientBoostingRegressor(
    n_estimators=200, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE
)
gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)

evaluate("Gradient Boosting", y_test, y_pred_gb, results)

### 5.6 Alternatives plus performantes : binning en histogrammes

`GradientBoostingRegressor` construit ses arbres sur les valeurs brutes des features,
ce qui devient lent à mesure que le nombre d'observations augmente. Les trois modèles
suivants discrétisent d'abord chaque variable en un nombre fixe de "bins" (256 par défaut),
ce qui accélère considérablement la recherche des meilleurs seuils de split — tout en
conservant, voire en améliorant, la performance prédictive.

- **`HistGradientBoostingRegressor`** : implémentation native scikit-learn, inspirée de
  LightGBM, sans dépendance externe.
- **`XGBoost`** : la référence historique du gradient boosting optimisé, très utilisée en
  compétition (Kaggle) et en production.
- **`LightGBM`** : développé par Microsoft, généralement le plus rapide des trois sur de
  gros volumes grâce à une croissance des arbres *leaf-wise* plutôt que *level-wise*.

```bash
pip install xgboost lightgbm
```

#### 5.6.1 HistGradientBoostingRegressor (scikit-learn natif)

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

hgb_model = HistGradientBoostingRegressor(
    max_iter=300,          # équivalent de n_estimators pour ce modèle
    learning_rate=0.05,
    max_depth=None,        # laisse l'algorithme choisir, régulé par max_leaf_nodes
    max_leaf_nodes=31,     # valeur par défaut, bon compromis biais/variance
    random_state=RANDOM_STATE,
)
hgb_model.fit(X_train, y_train)
y_pred_hgb = hgb_model.predict(X_test)

evaluate("HistGradientBoosting", y_test, y_pred_hgb, results)

#### 5.6.2 XGBoost

In [ ]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,          # échantillonne 80% des lignes par arbre : limite le surapprentissage
    colsample_bytree=0.8,   # idem sur les colonnes
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

evaluate("XGBoost", y_test, y_pred_xgb, results)

#### 5.6.3 LightGBM

In [ ]:
from lightgbm import LGBMRegressor

lgbm_model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,          # équivalent de max_leaf_nodes ; paramètre clé de LightGBM
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,             # supprime les logs d'entraînement verbeux par défaut
)
lgbm_model.fit(X_train, y_train)
y_pred_lgbm = lgbm_model.predict(X_test)

evaluate("LightGBM", y_test, y_pred_lgbm, results)

> ⚠️ **Sur un dataset de seulement ~20 000 lignes comme celui-ci**, l'écart de vitesse
> entre ces trois modèles et `GradientBoostingRegressor` sera peu perceptible : leur
> avantage se révèle surtout à partir de centaines de milliers, voire millions de lignes
> (comme votre dataset NYC Yellow Taxi, par exemple). Sur ce dataset-ci, jugez-les
> principalement sur la **performance** (MAE/RMSE/R²), pas sur la vitesse.

### 5.7 Comparaison de l'ensemble des modèles

In [ ]:
results_df = pd.DataFrame(results).sort_values("RMSE")
results_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, metric in zip(axes, ["MAE", "RMSE", "R2"]):
    ax.barh(results_df["Model"], results_df[metric])
    ax.set_title(metric)
    ax.invert_yaxis()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

Les modèles à base d'arbres (Random Forest, Gradient Boosting et les trois variantes à
binning) l'emportent nettement sur les modèles linéaires, ce qui suggère des relations non
linéaires et des interactions entre variables (confirmé par la visualisation géographique en
2.3 : la localisation module l'effet du revenu sur le prix).

**Regardez `results_df` trié par RMSE ci-dessus pour voir lequel de ces cinq modèles à base
d'arbres arrive en tête chez vous** — l'écart entre eux est généralement faible et peut varier
selon la machine. La suite du notebook poursuit l'optimisation sur **Random Forest** à titre
d'exemple pédagogique (il reste compétitif), mais vous pouvez répliquer la même démarche
de `RandomizedSearchCV` sur n'importe lequel des trois nouveaux modèles si l'un d'eux
s'avère meilleur dans vos résultats.

<a id='6'></a>
## 6. Optimisation des hyperparamètres

### 6.1 Validation croisée du modèle par défaut

Avant de lancer une recherche d'hyperparamètres coûteuse, on mesure la stabilité du modèle
actuel avec une validation croisée à 5 plis — un split train/test unique peut donner une
estimation optimiste ou pessimiste par pur hasard.

In [ ]:
cv_results_rf = cross_validate(
    rf_model, X_train, y_train, cv=5,
    scoring={"MAE": "neg_mean_absolute_error",
             "RMSE": "neg_root_mean_squared_error",
             "R2": "r2"},
    n_jobs=-1,
)

# scikit-learn utilise des scores "neg_*" par convention (plus grand = meilleur,
# pour que GridSearchCV/RandomizedSearchCV puissent toujours maximiser) : on les
# repasse en positif pour l'interprétation.
print(f"MAE  moyen : {-cv_results_rf['test_MAE'].mean():.4f} (± {cv_results_rf['test_MAE'].std():.4f})")
print(f"RMSE moyen : {-cv_results_rf['test_RMSE'].mean():.4f} (± {cv_results_rf['test_RMSE'].std():.4f})")
print(f"R²   moyen : {cv_results_rf['test_R2'].mean():.4f} (± {cv_results_rf['test_R2'].std():.4f})")

### 6.2 RandomizedSearchCV

Recherche aléatoire plutôt qu'exhaustive (`GridSearchCV`) : avec 4×4×3×3×2 = 288 combinaisons possibles, tester 20 tirages aléatoires sur 5 plis (100 entraînements) donne un bon compromis coût/performance.

In [ ]:
param_distributions = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": [1.0, "sqrt"],
}

rf_random_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=param_distributions,
    n_iter=20,
    cv=5,
    scoring="neg_root_mean_squared_error",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

rf_random_search.fit(X_train, y_train)

print("Meilleurs paramètres :", rf_random_search.best_params_)
print(f"Meilleur RMSE (CV)   : {-rf_random_search.best_score_:.4f}")

In [ ]:
best_rf = rf_random_search.best_estimator_
y_pred_best_rf = best_rf.predict(X_test)

evaluate("Random Forest (tuned)", y_test, y_pred_best_rf, results)

<a id='7'></a>
## 7. Interprétabilité

### 7.1 Feature importance (Random Forest)

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": best_rf.feature_importances_,
}).sort_values("Importance", ascending=False)

feature_importance

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(feature_importance["Feature"], feature_importance["Importance"])
plt.gca().invert_yaxis()
plt.xlabel("Importance")
plt.title("Importance des variables — Random Forest")
plt.savefig(f"{OUTPUT_DIR}/feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

### 7.2 Valeurs SHAP

La feature importance native de Random Forest indique *quelles* variables comptent en
moyenne, mais pas *comment* (positivement ou négativement) ni *pour quelle observation*.
SHAP répond à ces deux questions, au prix d'un calcul plus coûteux — d'où l'échantillonnage
à 500 observations plutôt que l'ensemble du jeu de test.

In [ ]:
X_shap = X_test.sample(n=500, random_state=RANDOM_STATE)

explainer = shap.TreeExplainer(best_rf)
shap_values = explainer.shap_values(X_shap)

print(f"Forme des valeurs SHAP : {shap_values.shape}")  # (n_observations, n_features)

In [ ]:
plt.figure()
shap.summary_plot(shap_values, X_shap, show=False)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/shap_summary.png", dpi=300, bbox_inches="tight")
plt.show()

<a id='8'></a>
## 8. Analyse des erreurs

In [ ]:
errors = pd.DataFrame({"Actual": y_test, "Predicted": y_pred_best_rf})
errors["Residual"] = errors["Actual"] - errors["Predicted"]
errors["AbsError"] = errors["Residual"].abs()
errors = errors.join(X_test)

# Les plus grosses erreurs se concentrent généralement sur les logements proches
# du plafond de 5.0 (voir l'avertissement en introduction) : le modèle ne peut pas
# prédire au-delà de ce qu'il a vu à l'entraînement.
errors.sort_values("AbsError", ascending=False).head(10)

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_best_rf, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
         linestyle="--", color="red", label="Prédiction parfaite")
plt.xlabel("Valeurs réelles")
plt.ylabel("Valeurs prédites")
plt.title("Valeurs réelles vs prédictions — Random Forest (tuned)")
plt.legend()
plt.savefig(f"{OUTPUT_DIR}/actual_vs_predicted.png", dpi=300, bbox_inches="tight")
plt.show()

<a id='9'></a>
## 9. Export pour déploiement

Pour servir ce modèle via une API, deux contraintes à respecter :

1. **Le feature engineering (§3) doit être reproduit à l'identique** au moment de la
   prédiction — si l'API reçoit les 8 variables brutes du recensement, elle doit recalculer
   `rooms_per_household`, `bedrooms_per_room` et `population_per_household` exactement comme
   à l'entraînement. Dupliquer cette logique en Python dans l'API (copier-coller du code)
   est une source classique de bugs silencieux si le notebook évolue sans que l'API suive.
2. **Solution : un `Pipeline` scikit-learn unique**, qui embarque le feature engineering
   *et* le modèle dans un seul objet sérialisé. L'API n'a plus qu'à charger ce fichier et
   appeler `.predict()` sur les données brutes — aucune logique métier dupliquée.

Le code ci-dessous construit ce pipeline et l'exporte avec `joblib`. Le projet complet de
déploiement (API FastAPI + page de démo interactive) est fourni séparément — voir le README
du dossier `california_housing_api/`.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin


class FeatureEngineer(BaseEstimator, TransformerMixin):
    """Recalcule les 3 variables dérivées du §3 à partir des 8 variables brutes.

    Doit être définie dans un module importable (pas dans le notebook directement)
    pour que joblib puisse recharger l'objet dans un autre processus (l'API) --
    voir model_utils.py dans le projet de déploiement.
    """

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X["rooms_per_household"] = X["AveRooms"] / X["AveOccup"]
        X["bedrooms_per_room"] = X["AveBedrms"] / X["AveRooms"]
        X["population_per_household"] = X["Population"] / X["AveOccup"]
        return X


deployment_pipeline = Pipeline([
    ("feature_engineering", FeatureEngineer()),
    ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1,
                                     **rf_random_search.best_params_)),
])

# On réentraîne sur l'ensemble des données disponibles (train + test) : en production,
# contrairement à l'évaluation, on veut exploiter toute l'information disponible.
X_full_raw = df[housing.feature_names]  # les 8 colonnes brutes, sans les features dérivées
y_full = df["MedHouseVal"]

deployment_pipeline.fit(X_full_raw, y_full)

import joblib
joblib.dump(deployment_pipeline, f"{OUTPUT_DIR}/model.joblib")
print(f"Pipeline exporté -> {OUTPUT_DIR}/model.joblib")

> ⚠️ **Important** : `FeatureEngineer` doit être définie dans le même module (`model_utils.py`)
> aussi bien au moment de l'export qu'au moment du chargement par l'API — joblib sérialise
> une référence à la classe, pas son code. Si vous exportez depuis ce notebook tel quel,
> `joblib.load()` échouera côté API avec une erreur `AttributeError: Can't get attribute
> 'FeatureEngineer'`. Utilisez plutôt `train_export_model.py` fourni dans le projet
> `california_housing_api/`, qui importe cette classe proprement depuis `model_utils.py`.

<a id='10'></a>
## 10. Conclusion

**Modèle retenu : Random Forest (hyperparamètres optimisés par RandomizedSearchCV).**

| Point | Détail |
|---|---|
| Performance | Meilleur RMSE et R² parmi tous les modèles testés (voir tableau comparatif §5.7) |
| Variables clés | `MedInc` domine largement, suivi par la localisation géographique (`Latitude`/`Longitude`) |
| Limite principale | Le plafonnement de `MedHouseVal` à 5.0 biaise structurellement les prédictions sur les logements haut de gamme — ce n'est pas une faiblesse du modèle mais du dataset source |
| Robustesse | Validation croisée à 5 plis confirmant la stabilité des résultats (écart-type faible) |

**Pistes d'amélioration pour aller plus loin :**
- Ajouter des features géospatiales dérivées (distance au centre-ville le plus proche, à la côte)
- Traiter explicitement les observations plafonnées (`MedHouseVal == 5.00001`) comme une classe à part, ou les exclure pour entraîner un modèle de régression plus propre
- Exposer d'autres modèles que Random Forest via l'API de déploiement (§9), pour comparer leurs prédictions en temps réel